# 08. 딥러닝 - 이진분류 (Keras)

> 다중분류(08_딥러닝_분류_다중.ipynb)와 출력층/손실함수가 달라 별도 파일로 분리했습니다. Titanic 생존 예측을 딥러닝으로 풀어봅니다.

### 이 노트북에서 배우는 것
- 이진분류 DL과 다중분류 DL이 무엇이 다른지 먼저 이해하기
- 출력층 노드 1개 + sigmoid 설계
- Dropout으로 과적합 막기, EarlyStopping+ModelCheckpoint 콜백 조합
- loss='binary_crossentropy' 컴파일
- loss curve, confusion matrix, ROC curve로 성능 확인
- 불균형 데이터에서 DL 재현율을 높이는 방법(SMOTE)

### 사용 데이터
- `data/train.csv` (Titanic) - Survived(0/1) 예측

## 목차
1. 이진분류 DL이란? 다중분류 DL과 무엇이 다른가
2. 이진분류용 Sequential 모델 설계
3. 모델 컴파일
4. 콜백 함수
5. 모델 학습
6. 성능 시각화
7. 불균형 데이터 & 재현율 개선 (DL)
8. 종합 실전 문제

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

tf.random.set_seed(100)

df = pd.read_csv('../data/train.csv')
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Embarked'] = df['Embarked'].fillna(df['Embarked'].mode()[0])
df['Sex'] = df['Sex'].map({'male': 1, 'female': 0})
df = pd.get_dummies(df, columns=['Embarked'], drop_first=True)

feature_cols = ['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare'] + [c for c in df.columns if c.startswith('Embarked_')]
X = df[feature_cols]
y = df['Survived']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)
X_train.shape

## 1. 이진분류 DL이란? 다중분류 DL과 무엇이 다른가

### 📖 이론 설명
`08_딥러닝_회귀.ipynb`(회귀)와 비교하면 **출력층**이 달라집니다. 이진분류는 '양성일 확률(0~1 사이 값)' 하나만 출력하면 되므로, 출력층은 **노드 1개 + activation='sigmoid'**입니다(sigmoid는 어떤 값이든 0~1 사이로 눌러주는 함수입니다). 다중분류(10번 노트북)는 클래스별 확률이 필요해 **노드 수=클래스 수 + activation='softmax'**를 씁니다. 손실함수도 이진분류는 `binary_crossentropy`, 다중분류는 `categorical_crossentropy`(또는 `sparse_categorical_crossentropy`)로 달라집니다. 이 노트북과 10번 노트북을 나란히 비교하면서 보면 차이가 뚜렷하게 보일 것입니다.

## 2. 이진분류용 Sequential 모델 설계

### 📖 이론 설명
출력층은 노드 1개, activation은 'sigmoid'로 고정합니다. `Dropout(rate)`는 학습 시 매 스텝마다 지정한 비율만큼의 노드 연결을 임의로 '끊어서' 특정 노드에 과도하게 의존하는 것을 막아 과적합을 줄여주는 정규화 기법입니다 - 추론(predict) 시에는 적용되지 않고 전체 노드를 다 사용합니다.

**BatchNormalization**은 Dropout과는 목적이 다릅니다 - 층을 지날 때마다 값의 분포가 들쭉날쭉해지는 '내부 공변량 변화(Internal Covariate Shift)' 문제를 줄이기 위해, 각 배치마다 값을 평균 0·분산 1 근처로 재정규화합니다. 그 결과 학습이 더 안정적이고 빨라지는 효과가 있어, 보통 `Dense` 층과 활성화 함수 사이에 넣습니다.

### 🧩 핵심 개념 정리
| 레이어 | 노드 수 | activation |
|---|---|---|
| 은닉층 | 자유 | 'relu' |
| BatchNormalization | - | 층 출력값을 재정규화(과적합 방지+학습 안정화) |
| Dropout | - | rate(0~1) 비율만큼 학습시 무작위 비활성화 |
| 출력층(이진분류) | 1 | 'sigmoid' |

### 💻 예제 코드

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization

model = Sequential()
model.add(Dense(16, activation='relu', input_shape=(X_train.shape[1],)))
model.add(Dropout(0.3))
model.add(Dense(8, activation='relu'))
model.add(Dropout(0.3))
model.add(Dense(1, activation='sigmoid'))
model.summary()

### ✍️ TODO: 실전 문제

**문제 1.** Dropout 없이, 은닉층 32→16(둘 다 relu), 출력층 1(sigmoid)인 `model_nodrop`을 만들어 `summary()`를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
model_nodrop = Sequential()
model_nodrop.add(Dense(32, activation='relu', input_shape=(X_train.shape[1],)))
model_nodrop.add(Dense(16, activation='relu'))
model_nodrop.add(Dense(1, activation='sigmoid'))  # 이진분류 출력층: 노드 1개 + sigmoid로 0~1 확률값을 냄
model_nodrop.summary()
```

</details>

**문제 2.** 은닉층 `Dense(32, relu)` 다음에 `BatchNormalization()`을 추가한 `model_bn`을 만들어 `summary()`를 확인하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
model_bn = Sequential()
model_bn.add(Dense(32, activation='relu', input_shape=(X_train.shape[1],)))
model_bn.add(BatchNormalization())  # Dense와 활성화 함수 사이에 넣어 층 출력값을 재정규화(학습 안정화)
model_bn.add(Dense(16, activation='relu'))
model_bn.add(Dense(1, activation='sigmoid'))
model_bn.summary()
```

</details>

## 3. 모델 컴파일

### 📖 이론 설명
이진분류의 손실함수는 `binary_crossentropy`이고, 평가지표는 보통 `accuracy`를 함께 봅니다.

### 🧩 핵심 개념 정리
| 옵션 | 이진분류에서의 값 |
|---|---|
| loss | 'binary_crossentropy' |
| metrics | ['accuracy'] |

### 💻 예제 코드

In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

### ✍️ TODO: 실전 문제

**문제 1.** `model_nodrop`을 동일한 옵션으로 컴파일하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
model_nodrop.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])  # 이진분류의 표준 loss는 binary_crossentropy
```

</details>

## 4. 콜백 함수

### 📖 이론 설명
`EarlyStopping`은 성능 개선이 멈추면 학습을 조기 종료하고, `ModelCheckpoint`는 **val_loss가 가장 낮았던 순간의 모델을 파일로 저장**해둡니다. 두 콜백을 리스트로 함께 넘기면, '가장 좋았던 모델을 놓치지 않고 저장하면서 동시에 불필요한 학습 시간도 줄이는' 조합이 됩니다.

### 🧩 핵심 개념 정리
| 콜백 | 역할 |
|---|---|
| EarlyStopping(monitor, patience) | 개선 없으면 학습 조기 종료 |
| ModelCheckpoint(path, monitor, save_best_only) | 가장 좋은 모델을 파일로 저장 |

### 💻 예제 코드

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(monitor='val_loss', mode='min', patience=10, restore_best_weights=True)
check_point = ModelCheckpoint('best_binary_model.h5', monitor='val_loss', mode='min', save_best_only=True)
callbacks = [early_stop, check_point]

## 5. 모델 학습

### 📖 이론 설명
콜백 리스트를 `fit`의 `callbacks` 인자로 전달합니다.

### 💻 예제 코드

In [ ]:
history = model.fit(X_train, y_train, epochs=100, batch_size=32,
                     validation_data=(X_test, y_test), callbacks=callbacks, verbose=0)
print('학습된 epoch 수:', len(history.history['loss']))

### ✍️ TODO: 실전 문제

**문제 1.** `model_nodrop`을 `EarlyStopping(patience=10)`만 사용해 `epochs=100`으로 학습시키세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
es2 = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
history2 = model_nodrop.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[es2], verbose=0)  # epochs=100을 최댓값으로 주고, 실제로는 EarlyStopping이 알아서 적절한 시점에 멈춰줌
print(len(history2.history['loss']))
```

</details>

## 6. 성능 시각화

### 📖 이론 설명
이진분류 모델은 `predict()`로 0~1 사이 확률을 출력하므로, **0.5를 임계값으로 0/1 클래스로 변환**해서 confusion matrix나 classification_report에 넣어야 합니다.

### 🧩 핵심 개념 정리
| 코드 | 설명 |
|---|---|
| (pred > 0.5).astype(int) | 확률 → 0/1 클래스 변환 |
| roc_curve(y_true, pred) | 임계값을 바꿔가며 성능 확인 |

### 💻 예제 코드

In [ ]:
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score, classification_report

plt.plot(history.history['loss'], label='train loss')
plt.plot(history.history['val_loss'], label='val loss')
plt.xlabel('Epoch'); plt.ylabel('Loss'); plt.title('Loss Curve'); plt.legend(); plt.show()

pred_proba = model.predict(X_test).flatten()
pred_label = (pred_proba > 0.5).astype(int)

cm = confusion_matrix(y_test, pred_label)
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds')
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('DL Binary Classification Confusion Matrix')
plt.show()

fpr, tpr, _ = roc_curve(y_test, pred_proba)
plt.plot(fpr, tpr, label=f'AUC={roc_auc_score(y_test, pred_proba):.3f}')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR'); plt.ylabel('TPR'); plt.title('ROC Curve'); plt.legend(); plt.show()

print(classification_report(y_test, pred_label))

### ✍️ TODO: 실전 문제

**문제 1.** `model_nodrop`의 예측 확률을 0.5 기준으로 0/1로 변환한 뒤 accuracy를 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
from sklearn.metrics import accuracy_score
pred2 = model_nodrop.predict(X_test).flatten()  # predict()는 0~1 사이 확률을 반환(아직 클래스가 아님)
pred2_label = (pred2 > 0.5).astype(int)  # 0.5를 기준으로 확률을 0/1 클래스로 변환
print(accuracy_score(y_test, pred2_label))
```

</details>

## 7. 불균형 데이터 & 재현율 개선 (DL)

### 📖 이론 설명
머신러닝(06번)에서 다룬 것과 같은 문제가 딥러닝에서도 나타납니다 - 소수 클래스 데이터가 적으면 모델이 그 클래스를 잘 예측하지 못해 재현율이 낮게 나옵니다. `imblearn`의 `SMOTE`로 학습 데이터를 오버샘플링한 뒤 다시 학습시키면 재현율이 개선되는지 확인해봅니다(단, 정밀도가 함께 떨어질 수 있다는 트레이드오프는 동일하게 적용됩니다).

### 💻 예제 코드

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.metrics import recall_score, precision_score

print('SMOTE 적용 전:', pd.Series(y_train).value_counts().to_dict())
smote = SMOTE(random_state=42)
X_train_over, y_train_over = smote.fit_resample(X_train, y_train)
print('SMOTE 적용 후:', pd.Series(y_train_over).value_counts().to_dict())

model_smote = Sequential([
    Dense(16, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(8, activation='relu'),
    Dense(1, activation='sigmoid'),
])
model_smote.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_smote.fit(X_train_over, y_train_over, epochs=50, batch_size=32, verbose=0)

pred_smote = (model_smote.predict(X_test).flatten() > 0.5).astype(int)
print('SMOTE 적용 후 recall:', recall_score(y_test, pred_smote))
print('SMOTE 적용 후 precision:', precision_score(y_test, pred_smote))

## 8. 종합 실전 문제

**딥러닝 이진분류 전체 흐름을 이어서 풀어보는 미니 모의고사입니다.**

**문제 1.** 입력층 64(relu)+Dropout(0.3)→32(relu)+Dropout(0.3)→출력 1(sigmoid) 구조의 `model_final`을 만들고 `binary_crossentropy`로 컴파일하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
model_final = Sequential()
model_final.add(Dense(64, activation='relu', input_shape=(X_train.shape[1],)))
model_final.add(Dropout(0.3))  # 학습 시 30% 노드 연결을 무작위로 꺼서 특정 노드에 과도하게 의존하는 것을 막음(과적합 방지)
model_final.add(Dense(32, activation='relu'))
model_final.add(Dropout(0.3))
model_final.add(Dense(1, activation='sigmoid'))
model_final.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model_final.summary()
```

</details>

**문제 2.** `model_final`을 `EarlyStopping(patience=10)`으로 학습시키고, 테스트 데이터의 accuracy를 `evaluate()`로 구하세요.

In [ ]:
# 여기에 코드를 작성하세요


<details>
<summary>✅ 정답 보기</summary>

```python
es_final = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
model_final.fit(X_train, y_train, epochs=100, batch_size=32, validation_data=(X_test, y_test), callbacks=[es_final], verbose=0)
loss, acc = model_final.evaluate(X_test, y_test, verbose=0)  # evaluate()는 compile 시 지정한 순서대로 (loss, metrics...)를 반환
print(acc)
```

</details>